# Forward-Looking Backtest: Calendar-Time Cutoff Comparison

This notebook evaluates the **predictive power** of three competing-risk models by
projecting cash flows forward from calendar-time cutoff dates and comparing against
realized outcomes.

**No data leakage**: At each cutoff date, all models are **retrained** using only
data available before the cutoff. Predictions use only cutoff-date features.

## Models

| Model | Type | Training data | Prediction input |
|-------|------|---------------|------------------|
| **Aalen-Johansen** | Nonparametric CIF | Pre-cutoff loan histories | Population-level (no features) |
| **Cox TV** | Semi-parametric | Pre-cutoff loan-month panel | Frozen macro + deterministic bal_repaid, t_act_12m |
| **RSF** | ML ensemble | Pre-cutoff terminal observations | Snapshot features at cutoff |

## Feature handling after cutoff

- **Evolve deterministically**: `bal_repaid` (amortization), `t_act_12m` (min(12, age))
- **Freeze at cutoff value**: All 12 macro features, `t_del_30d_12m`, `t_del_60d_12m`
- **Always static**: `int_rate`, `orig_upb`/`log_upb`, `fico_score`, `dti_r`, `ltv_r`

## Cutoff dates

2019-06, 2020-06, 2021-06, 2022-06, 2023-06

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from lifelines import AalenJohansenFitter, CoxTimeVaryingFitter
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

import sys
sys.path.insert(0, '..')

from src.alm.baseline_hazard import extract_baseline_hazards_both
from src.alm.cash_flow_engine import CashFlowConfig, MortgageCashFlowEngine
from src.alm.rsf_cash_flow_engine import RSFCashFlowConfig, RSFCashFlowEngine

sns.set_style('whitegrid')
%matplotlib inline

DATA_DIR = Path('../data/processed')
EXTERNAL_DIR = Path('../data/external')
FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

print('Imports complete.')

In [ ]:
# Cutoff dates and evaluation horizons
CUTOFF_DATES = [
    pd.Period('2019-06', 'M'),
    pd.Period('2020-06', 'M'),
    pd.Period('2021-06', 'M'),
    pd.Period('2022-06', 'M'),
    pd.Period('2023-06', 'M'),
]
HORIZONS = [12, 24, 36]  # months post-cutoff to evaluate
MAX_HORIZON = 36
LGD = 0.25

print(f'Cutoff dates: {[str(c) for c in CUTOFF_DATES]}')
print(f'Evaluation horizons: {HORIZONS} months')

In [ ]:
# Feature definitions (matching notebooks 05 and 07)
COX_FEATURE_NAMES = [
    'int_rate', 'orig_upb', 'fico_score', 'dti_r', 'ltv_r',
    'bal_repaid', 't_act_12m', 't_del_30d_12m', 't_del_60d_12m',
    'hpi_st_d_t_o', 'ppi_c_FRMA', 'TB10Y_d_t_o', 'FRMA30Y_d_t_o',
    'ppi_o_FRMA', 'hpi_st_log12m', 'hpi_r_st_us', 'st_unemp_r12m',
    'st_unemp_r3m', 'TB10Y_r12m', 'T10Y3MM', 'T10Y3MM_r12m',
]

# RSF uses log_upb instead of orig_upb and bal_repaid_lag1 instead of bal_repaid
RSF_FEATURE_COLS = [
    'int_rate', 'log_upb', 'fico_score', 'dti_r', 'ltv_r',
    'bal_repaid_lag1', 't_act_12m', 't_del_30d_12m', 't_del_60d_12m',
    'hpi_st_d_t_o', 'ppi_c_FRMA', 'TB10Y_d_t_o', 'FRMA30Y_d_t_o',
    'ppi_o_FRMA', 'hpi_st_log12m', 'hpi_r_st_us', 'st_unemp_r12m',
    'st_unemp_r3m', 'TB10Y_r12m', 'T10Y3MM', 'T10Y3MM_r12m',
]

# RSF hyperparameters (matching notebook 07)
RSF_PARAMS = {
    'n_estimators': 100,
    'max_depth': 10,
    'min_samples_split': 20,
    'min_samples_leaf': 30,
    'max_features': 2,
    'n_jobs': -1,
    'random_state': 42,
}

# Cox penalizer (matching notebook 05)
COX_PENALIZER = 0.01

print(f'Cox features ({len(COX_FEATURE_NAMES)})')
print(f'RSF features ({len(RSF_FEATURE_COLS)})')
print(f'RSF params: {RSF_PARAMS}')

In [ ]:
# Load panel and supporting data
panel_df = pd.read_parquet(DATA_DIR / 'loan_month_panel.parquet')
surv_df = pd.read_parquet(DATA_DIR / 'survival_data_blumenstock.parquet')

# Join orig_loan_term and origination-time macro onto panel
orig_cols = ['loan_sequence_number', 'orig_loan_term', 'orig_MORTGAGE30US', 'orig_DGS10', 'orig_state_hpi']
orig_info = surv_df[orig_cols].drop_duplicates(subset=['loan_sequence_number'])

print(f'Panel: {len(panel_df):,} loan-months, {panel_df["loan_sequence_number"].nunique():,} loans')
print(f'Date range: {panel_df["year_month"].min()} to {panel_df["year_month"].max()}')

---

## Helper Functions

1. **`get_active_loans_at_cutoff`**: Identify loans still active at the cutoff date
2. **`train_cox_models`**: Train Cox TV models on pre-cutoff panel data
3. **`train_rsf_models`**: Train RSF models on pre-cutoff terminal observations
4. **`build_frozen_cox_matrix`**: Build 3D covariate matrix with frozen macro
5. **`build_rsf_snapshot`**: Build 2D feature matrix for RSF
6. **`compute_realized`**: Extract realized CIF from post-cutoff panel
7. **`fit_conditional_aj`**: Fit Aalen-Johansen CIF on pre-cutoff data

In [ ]:
def get_active_loans_at_cutoff(panel_df, orig_info, cutoff):
    """
    Get all loans active at the cutoff date with their feature values.
    A loan is active if it has an observation at cutoff with event == 0.

    For bal_repaid_lag1, we use the previous month's bal_repaid to match
    the lag-1 convention used in RSF training.
    """
    cutoff_rows = panel_df[
        (panel_df['year_month'] == cutoff) & (panel_df['event'] == 0)
    ].copy()

    # Join origination info
    cutoff_rows = cutoff_rows.merge(orig_info, on='loan_sequence_number', how='left')
    cutoff_rows['orig_loan_term'] = cutoff_rows['orig_loan_term'].fillna(360).astype(int)
    cutoff_rows['current_loan_age'] = cutoff_rows['loan_age'].astype(int)
    cutoff_rows['log_upb'] = np.log(cutoff_rows['orig_upb'].astype(float))

    # bal_repaid_lag1: use previous month's bal_repaid (matching RSF training convention)
    prev_month = cutoff - 1
    prev_rows = panel_df[
        (panel_df['year_month'] == prev_month)
    ][['loan_sequence_number', 'bal_repaid']].rename(
        columns={'bal_repaid': 'bal_repaid_lag1'}
    )
    cutoff_rows = cutoff_rows.merge(prev_rows, on='loan_sequence_number', how='left')
    # Fallback: if no previous month row exists, use current bal_repaid
    cutoff_rows['bal_repaid_lag1'] = cutoff_rows['bal_repaid_lag1'].fillna(
        cutoff_rows['bal_repaid']
    ).astype(float)

    return cutoff_rows

In [ ]:
def train_cox_models(panel_df, cutoff, feature_names, penalizer=0.01):
    """
    Train cause-specific Cox TV models on all panel data up to the cutoff.

    Returns (cox_engine, n_train_loans, n_train_months)
    """
    # Filter to pre-cutoff data only
    train_panel = panel_df[panel_df['year_month'] <= cutoff].copy()
    train_panel = train_panel.dropna(subset=feature_names).copy()

    n_loans = train_panel['loan_sequence_number'].nunique()
    n_months = len(train_panel)

    # Prepayment: event = 1 for prepay, 0 for default or censored
    train_panel['event_prepay'] = (
        (train_panel['event'] == 1) & (train_panel['event_code'] == 1)
    ).astype(int)

    # Default: event = 1 for default, 0 for prepay or censored
    train_panel['event_default'] = (
        (train_panel['event'] == 1) & (train_panel['event_code'] == 2)
    ).astype(int)

    cox_cols_p = ['loan_sequence_number', 'start', 'stop', 'event_prepay'] + feature_names
    cox_cols_d = ['loan_sequence_number', 'start', 'stop', 'event_default'] + feature_names

    # Fit prepay model
    ctv_prepay = CoxTimeVaryingFitter(penalizer=penalizer)
    ctv_prepay.fit(
        train_panel[cox_cols_p],
        id_col='loan_sequence_number',
        start_col='start',
        stop_col='stop',
        event_col='event_prepay',
        show_progress=False,
    )

    # Fit default model
    ctv_default = CoxTimeVaryingFitter(penalizer=penalizer)
    ctv_default.fit(
        train_panel[cox_cols_d],
        id_col='loan_sequence_number',
        start_col='start',
        stop_col='stop',
        event_col='event_default',
        show_progress=False,
    )

    # Build engine
    h0_prepay, h0_default = extract_baseline_hazards_both(ctv_prepay, ctv_default)
    beta_prepay = ctv_prepay.params_.values.astype(np.float64)
    beta_default = ctv_default.params_.values.astype(np.float64)

    engine = MortgageCashFlowEngine(
        h0_prepay, h0_default, beta_prepay, beta_default,
        config=CashFlowConfig(lgd=LGD, projection_horizon=MAX_HORIZON),
    )

    n_prepay = train_panel['event_prepay'].sum()
    n_default = train_panel['event_default'].sum()

    return engine, ctv_prepay.params_.index.tolist(), n_loans, n_months, n_prepay, n_default

In [ ]:
def train_rsf_models(panel_df, cutoff, rsf_feature_cols, rsf_params):
    """
    Train cause-specific RSF models on terminal observations from pre-cutoff data.

    For each loan observed before cutoff:
    - If it terminated (prepay/default) before cutoff: use terminal observation
    - If still active at cutoff: use last observation, censored (event_code=0)

    Returns (rsf_engine, n_train_loans, n_prepay, n_default)
    """
    # Get pre-cutoff panel
    before = panel_df[panel_df['year_month'] <= cutoff].copy()
    before = before.sort_values(['loan_sequence_number', 'loan_age'])

    # Last observation per loan (terminal or censored-at-cutoff)
    terminal_df = before.groupby('loan_sequence_number').last().reset_index()

    # For censored loans (still active at cutoff), set event_code = 0
    # This is already correct: if their last row before cutoff has event=0,
    # event_code is already 0. If event=1, they terminated before cutoff.

    # Lag bal_repaid to avoid leakage (matching notebook 07)
    def get_lagged_bal_repaid(group):
        if len(group) >= 2:
            return group['bal_repaid'].iloc[-2]
        else:
            return group['bal_repaid'].iloc[-1]

    bal_repaid_lag = before.groupby('loan_sequence_number').apply(get_lagged_bal_repaid)
    terminal_df['bal_repaid_lag1'] = terminal_df['loan_sequence_number'].map(bal_repaid_lag)

    # Log transform UPB
    terminal_df['log_upb'] = np.log(terminal_df['orig_upb'].astype(float))

    # Drop NaN features
    terminal_df = terminal_df.dropna(subset=rsf_feature_cols).copy()

    X_train = terminal_df[rsf_feature_cols].values
    duration = terminal_df['loan_age'].values.astype(float)
    event_code = terminal_df['event_code'].values

    n_loans = len(terminal_df)
    n_prepay = int((event_code == 1).sum())
    n_default = int((event_code == 2).sum())

    # Prepayment RSF
    y_prepay = Surv.from_arrays(event_code == 1, duration)
    rsf_p = RandomSurvivalForest(**rsf_params)
    rsf_p.fit(X_train, y_prepay)

    # Default RSF
    y_default = Surv.from_arrays(event_code == 2, duration)
    rsf_d = RandomSurvivalForest(**rsf_params)
    rsf_d.fit(X_train, y_default)

    engine = RSFCashFlowEngine(
        rsf_p, rsf_d,
        config=RSFCashFlowConfig(lgd=LGD, projection_horizon=MAX_HORIZON),
    )

    return engine, n_loans, n_prepay, n_default

In [ ]:
def build_frozen_cox_matrix(active_loans, feature_names, horizon):
    """
    Build 3D covariate matrix (N, T, F) for Cox engine.

    - Deterministic: bal_repaid (amortization), t_act_12m (min(12, age))
    - Frozen at cutoff: all macro features, delinquency counts
    - Static: int_rate, orig_upb, fico_score, dti_r, ltv_r
    """
    N = len(active_loans)
    F = len(feature_names)
    feat_idx = {name: i for i, name in enumerate(feature_names)}

    X = np.zeros((N, horizon, F), dtype=np.float32)

    # Tile cutoff-row feature values across all months (frozen baseline)
    cutoff_features = active_loans[feature_names].values.astype(np.float32)
    for t in range(horizon):
        X[:, t, :] = cutoff_features

    # Overwrite deterministic features that evolve
    int_rate = active_loans['int_rate'].values.astype(np.float32)
    orig_upb = active_loans['orig_upb'].values.astype(np.float32)
    term = active_loans['orig_loan_term'].values.astype(np.float32)
    current_age = active_loans['current_loan_age'].values.astype(np.float32)

    # Amortization: compute bal_repaid at each future month
    monthly_rate = int_rate / 100.0 / 12.0
    payment = np.where(
        monthly_rate > 0,
        orig_upb * monthly_rate / (1.0 - (1.0 + monthly_rate) ** (-term)),
        orig_upb / term,
    )

    if 'bal_repaid' in feat_idx:
        for t in range(horizon):
            n_payments = current_age + t + 1
            factor = (1.0 + monthly_rate) ** n_payments
            upb_t = np.where(
                monthly_rate > 0,
                orig_upb * factor - payment * (factor - 1.0) / np.where(monthly_rate > 0, monthly_rate, 1.0),
                orig_upb - payment * n_payments,
            )
            upb_t = np.maximum(upb_t, 0.0)
            bal_repaid_t = (orig_upb - upb_t) / orig_upb * 100.0
            X[:, t, feat_idx['bal_repaid']] = bal_repaid_t

    # t_act_12m: deterministic, assuming performing
    if 't_act_12m' in feat_idx:
        for t in range(horizon):
            loan_age = current_age + t + 1
            X[:, t, feat_idx['t_act_12m']] = np.minimum(12.0, loan_age)

    return X

In [ ]:
def build_rsf_snapshot(active_loans, rsf_feature_cols):
    """
    Build 2D feature matrix (N, F) for RSF from cutoff-date snapshot.
    """
    return active_loans[rsf_feature_cols].values.astype(np.float64)

In [ ]:
def compute_realized(panel_df, active_loan_ids, cutoff, horizon):
    """
    Compute realized CIF and cash flows for active loans after the cutoff.

    Returns dict with:
      - cif_prepay, cif_default: arrays of shape (horizon,)
      - n_prepay, n_default, n_censored: event counts
      - monthly_prepay_count, monthly_default_count: events per month
    """
    N = len(active_loan_ids)

    # Get all rows for active loans after cutoff
    mask = (
        panel_df['loan_sequence_number'].isin(active_loan_ids) &
        (panel_df['year_month'] > cutoff)
    )
    future = panel_df[mask].copy()

    # Months since cutoff
    future['months_since_cutoff'] = (future['year_month'] - cutoff).apply(lambda x: x.n)

    # Find terminal events for each loan (first row with event == 1)
    events = future[future['event'] == 1].groupby('loan_sequence_number').first()

    # Count events by months-since-cutoff
    prepay_counts = np.zeros(horizon)
    default_counts = np.zeros(horizon)

    for _, row in events.iterrows():
        m = int(row['months_since_cutoff']) - 1  # 0-indexed
        if m < 0 or m >= horizon:
            continue
        if row['event_code'] == 1:
            prepay_counts[m] += 1
        elif row['event_code'] == 2:
            default_counts[m] += 1

    cif_prepay = np.cumsum(prepay_counts) / N
    cif_default = np.cumsum(default_counts) / N

    return {
        'cif_prepay': cif_prepay,
        'cif_default': cif_default,
        'n_prepay': int(prepay_counts.sum()),
        'n_default': int(default_counts.sum()),
        'n_active': N,
        'monthly_prepay_count': prepay_counts,
        'monthly_default_count': default_counts,
    }

In [ ]:
def fit_conditional_aj(panel_df, active_loans, cutoff, horizon):
    """
    Fit Aalen-Johansen on loans active at cutoff and compute conditional
    forward-looking CIF.

    We use all loans that existed before the cutoff to estimate the
    unconditional CIF, then condition on survival to each loan's age
    at cutoff to get the forward CIF.

    CIF_forward(t | survived to age a) = [CIF(a+t) - CIF(a)] / S(a)
    """
    # Build training data: all loans with observations up to cutoff
    # For each loan, get its status as of the cutoff
    before = panel_df[panel_df['year_month'] <= cutoff]
    loan_last = before.groupby('loan_sequence_number').last().reset_index()

    # Duration = loan_age at last observation before/at cutoff
    # Event = actual event if terminal row is at/before cutoff, else 0 (censored at cutoff)
    duration = loan_last['loan_age'].values.astype(float)
    event_code = np.where(
        loan_last['event'] == 1,
        loan_last['event_code'].values,
        0,
    )

    # Fit AJ for prepay and default
    ajf_prepay = AalenJohansenFitter(calculate_variance=False)
    ajf_prepay.fit(duration, event_code, event_of_interest=1)

    ajf_default = AalenJohansenFitter(calculate_variance=False)
    ajf_default.fit(duration, event_code, event_of_interest=2)

    # Get CIF and survival at all time points
    cif_p = ajf_prepay.cumulative_density_.iloc[:, 0]
    cif_d = ajf_default.cumulative_density_.iloc[:, 0]

    # Overall survival: S(t) = 1 - CIF_prepay(t) - CIF_default(t)
    # Reindex both to a common grid
    max_t = int(max(cif_p.index.max(), cif_d.index.max())) + 1
    t_grid = np.arange(max_t + 1)
    cif_p_full = cif_p.reindex(t_grid).ffill().fillna(0.0).values
    cif_d_full = cif_d.reindex(t_grid).ffill().fillna(0.0).values
    surv_full = 1.0 - cif_p_full - cif_d_full

    # Conditional forward CIF: average across active loans
    # For each loan with age a at cutoff:
    #   CIF_forward_k(t) = [CIF_k(a+t) - CIF_k(a)] / S(a)
    ages = active_loans['current_loan_age'].values.astype(int)
    N = len(ages)

    fwd_cif_p = np.zeros((N, horizon))
    fwd_cif_d = np.zeros((N, horizon))

    for i, a in enumerate(ages):
        a = min(a, max_t)
        s_a = max(surv_full[a], 1e-10)
        for t in range(horizon):
            at = min(a + t + 1, max_t)
            fwd_cif_p[i, t] = max(0, (cif_p_full[at] - cif_p_full[a]) / s_a)
            fwd_cif_d[i, t] = max(0, (cif_d_full[at] - cif_d_full[a]) / s_a)

    return {
        'cif_prepay': fwd_cif_p.mean(axis=0),
        'cif_default': fwd_cif_d.mean(axis=0),
    }

---

## Main Computation: Loop Over Cutoff Dates

For each cutoff date:
1. **Retrain** Cox and RSF models on pre-cutoff data only (no data leakage)
2. Identify active loans and extract cutoff-date features
3. Fit nonparametric CIF (Aalen-Johansen) on pre-cutoff data
4. Project Cox model forward with frozen macro
5. Project RSF model forward with snapshot features
6. Extract realized outcomes from post-cutoff panel

In [ ]:
all_results = {}

for cutoff in CUTOFF_DATES:
    cutoff_str = str(cutoff)
    print(f'\n{"=" * 60}')
    print(f'CUTOFF: {cutoff_str}')
    print(f'{"=" * 60}')

    # 1. Active loans at cutoff
    active = get_active_loans_at_cutoff(panel_df, orig_info, cutoff)
    active = active.dropna(subset=COX_FEATURE_NAMES).copy()
    n_active = len(active)
    print(f'Active loans: {n_active:,}')
    print(f'Mean loan age: {active["current_loan_age"].mean():.1f} months')

    # 2. Train Cox models on pre-cutoff data
    print('\nTraining Cox models on pre-cutoff data...')
    cox_engine, cox_features, n_cox_loans, n_cox_months, n_cox_prepay, n_cox_default = \
        train_cox_models(panel_df, cutoff, COX_FEATURE_NAMES, COX_PENALIZER)
    print(f'  Training data: {n_cox_loans:,} loans, {n_cox_months:,} loan-months')
    print(f'  Events: {n_cox_prepay:,} prepays, {n_cox_default:,} defaults')

    # 3. Train RSF models on pre-cutoff data
    print('Training RSF models on pre-cutoff data...')
    rsf_engine, n_rsf_loans, n_rsf_prepay, n_rsf_default = \
        train_rsf_models(panel_df, cutoff, RSF_FEATURE_COLS, RSF_PARAMS)
    print(f'  Training data: {n_rsf_loans:,} loans')
    print(f'  Events: {n_rsf_prepay:,} prepays, {n_rsf_default:,} defaults')

    # 4. Nonparametric CIF (Aalen-Johansen on pre-cutoff data)
    print('Fitting Aalen-Johansen...')
    aj_result = fit_conditional_aj(panel_df, active, cutoff, MAX_HORIZON)
    print(f'  AJ CIF prepay at 12m: {aj_result["cif_prepay"][11]:.4f}')

    # 5. Cox forward projection
    print('Cox forward projection...')
    X_cox = build_frozen_cox_matrix(active, cox_features, MAX_HORIZON)
    cox_cf = cox_engine.project_cash_flows(active, X_cox)
    cox_cif_prepay = np.cumsum(cox_cf['f_prepay'], axis=1).mean(axis=0)
    cox_cif_default = np.cumsum(cox_cf['f_default'], axis=1).mean(axis=0)
    print(f'  Cox CIF prepay at 12m: {cox_cif_prepay[11]:.4f}')

    # 6. RSF forward projection
    print('RSF forward projection...')
    rsf_avail = [c for c in RSF_FEATURE_COLS if c in active.columns]
    X_rsf = build_rsf_snapshot(active, rsf_avail)
    rsf_cf = rsf_engine.project_cash_flows(active, X_rsf)
    rsf_cif_prepay = np.cumsum(rsf_cf['f_prepay'], axis=1).mean(axis=0)
    rsf_cif_default = np.cumsum(rsf_cf['f_default'], axis=1).mean(axis=0)
    print(f'  RSF CIF prepay at 12m: {rsf_cif_prepay[11]:.4f}')

    # 7. Realized outcomes
    print('Computing realized outcomes...')
    active_ids = set(active['loan_sequence_number'].values)
    realized = compute_realized(panel_df, active_ids, cutoff, MAX_HORIZON)
    print(f'  Realized: {realized["n_prepay"]:,} prepays, '
          f'{realized["n_default"]:,} defaults out of {realized["n_active"]:,} loans')
    print(f'  Realized CIF prepay at 12m: {realized["cif_prepay"][11]:.4f}')

    # Store results
    all_results[cutoff_str] = {
        'n_active': n_active,
        'cutoff': cutoff,
        'n_cox_train': n_cox_loans,
        'n_rsf_train': n_rsf_loans,
        'aj': aj_result,
        'cox_cif_prepay': cox_cif_prepay,
        'cox_cif_default': cox_cif_default,
        'cox_cf': cox_cf,
        'rsf_cif_prepay': rsf_cif_prepay,
        'rsf_cif_default': rsf_cif_default,
        'rsf_cf': rsf_cf,
        'realized': realized,
    }

print(f'\n{"=" * 60}')
print('All cutoffs complete.')

---

## CIF Comparison: Predicted vs Realized

In [ ]:
# Build comparison table: CIF at each horizon for each cutoff and model
rows = []
for cutoff_str, res in all_results.items():
    real = res['realized']
    for h in HORIZONS:
        if h > MAX_HORIZON:
            continue
        t = h - 1  # 0-indexed
        rows.append({
            'cutoff': cutoff_str,
            'horizon': h,
            'real_cif_p': real['cif_prepay'][t],
            'real_cif_d': real['cif_default'][t],
            'aj_cif_p': res['aj']['cif_prepay'][t],
            'aj_cif_d': res['aj']['cif_default'][t],
            'cox_cif_p': res['cox_cif_prepay'][t],
            'cox_cif_d': res['cox_cif_default'][t],
            'rsf_cif_p': res['rsf_cif_prepay'][t],
            'rsf_cif_d': res['rsf_cif_default'][t],
        })

cif_df = pd.DataFrame(rows)

# Compute errors
for model in ['aj', 'cox', 'rsf']:
    cif_df[f'{model}_err_p'] = cif_df[f'{model}_cif_p'] - cif_df['real_cif_p']
    cif_df[f'{model}_err_d'] = cif_df[f'{model}_cif_d'] - cif_df['real_cif_d']

# Print
print('=== CIF Comparison: Predicted vs Realized ===')
print()
for event, suffix in [('Prepayment', '_p'), ('Default', '_d')]:
    print(f'--- {event} CIF ---')
    cols = ['cutoff', 'horizon', f'real_cif{suffix}',
            f'aj_cif{suffix}', f'aj_err{suffix}',
            f'cox_cif{suffix}', f'cox_err{suffix}',
            f'rsf_cif{suffix}', f'rsf_err{suffix}']
    print(cif_df[cols].to_string(index=False, float_format='{:.4f}'.format))
    print()

---

## CIF Curves: Predicted vs Realized

In [ ]:
# Prepayment CIF curves
n_cutoffs = len(CUTOFF_DATES)
fig, axes = plt.subplots(1, n_cutoffs, figsize=(4 * n_cutoffs, 4), sharey=True)
months = np.arange(1, MAX_HORIZON + 1)

for i, (cutoff_str, res) in enumerate(all_results.items()):
    ax = axes[i]
    real = res['realized']

    ax.plot(months, real['cif_prepay'], 'k-', lw=2, label='Realized')
    ax.plot(months, res['aj']['cif_prepay'], '--', color='green', lw=1.5, label='AJ')
    ax.plot(months, res['cox_cif_prepay'], '--', color='steelblue', lw=1.5, label='Cox')
    ax.plot(months, res['rsf_cif_prepay'], '--', color='darkorange', lw=1.5, label='RSF')

    ax.set_title(f'Cutoff: {cutoff_str}')
    ax.set_xlabel('Months since cutoff')
    if i == 0:
        ax.set_ylabel('CIF Prepayment')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Prepayment CIF: Predicted vs Realized', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'backtest_cif_prepay.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Default CIF curves
fig, axes = plt.subplots(1, n_cutoffs, figsize=(4 * n_cutoffs, 4), sharey=True)

for i, (cutoff_str, res) in enumerate(all_results.items()):
    ax = axes[i]
    real = res['realized']

    ax.plot(months, real['cif_default'], 'k-', lw=2, label='Realized')
    ax.plot(months, res['aj']['cif_default'], '--', color='green', lw=1.5, label='AJ')
    ax.plot(months, res['cox_cif_default'], '--', color='steelblue', lw=1.5, label='Cox')
    ax.plot(months, res['rsf_cif_default'], '--', color='darkorange', lw=1.5, label='RSF')

    ax.set_title(f'Cutoff: {cutoff_str}')
    ax.set_xlabel('Months since cutoff')
    if i == 0:
        ax.set_ylabel('CIF Default')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Default CIF: Predicted vs Realized', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'backtest_cif_default.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Cash Flow Comparison

In [ ]:
# Cumulative total cash flow: predicted (Cox, RSF) vs realized
fig, axes = plt.subplots(1, n_cutoffs, figsize=(4 * n_cutoffs, 4), sharey=False)

for i, (cutoff_str, res) in enumerate(all_results.items()):
    ax = axes[i]
    real = res['realized']
    T = min(MAX_HORIZON, res['cox_cf']['total_cf'].shape[1])
    m = np.arange(1, T + 1)

    # Predicted cumulative CF (averaged across loans)
    cox_cumul = np.cumsum(res['cox_cf']['total_cf'].sum(axis=0)) / 1e6
    rsf_cumul = np.cumsum(res['rsf_cf']['total_cf'].sum(axis=0)) / 1e6

    # Realized cumulative CF: reconstruct from realized events
    # Use portfolio-level: interest + principal + prepay + recovery
    # For realized, compute from the panel data after cutoff
    active_ids = set(res['cox_cf']['total_cf'].shape[0] * [None])  # placeholder

    ax.plot(m[:T], cox_cumul[:T], '--', color='steelblue', lw=1.5, label='Cox')
    ax.plot(m[:T], rsf_cumul[:T], '--', color='darkorange', lw=1.5, label='RSF')

    ax.set_title(f'Cutoff: {cutoff_str}')
    ax.set_xlabel('Months since cutoff')
    if i == 0:
        ax.set_ylabel('Cumulative CF ($M)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Cumulative Cash Flows: Cox vs RSF', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'backtest_cumul_cf.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Survival Curves: Predicted vs Realized

In [ ]:
# Average survival S(t) = 1 - CIF_prepay(t) - CIF_default(t)
fig, axes = plt.subplots(1, n_cutoffs, figsize=(4 * n_cutoffs, 4), sharey=True)

for i, (cutoff_str, res) in enumerate(all_results.items()):
    ax = axes[i]
    real = res['realized']

    real_surv = 1.0 - real['cif_prepay'] - real['cif_default']
    aj_surv = 1.0 - res['aj']['cif_prepay'] - res['aj']['cif_default']
    cox_surv = 1.0 - res['cox_cif_prepay'] - res['cox_cif_default']
    rsf_surv = 1.0 - res['rsf_cif_prepay'] - res['rsf_cif_default']

    ax.plot(months, real_surv, 'k-', lw=2, label='Realized')
    ax.plot(months, aj_surv, '--', color='green', lw=1.5, label='AJ')
    ax.plot(months, cox_surv, '--', color='steelblue', lw=1.5, label='Cox')
    ax.plot(months, rsf_surv, '--', color='darkorange', lw=1.5, label='RSF')

    ax.set_title(f'Cutoff: {cutoff_str}')
    ax.set_xlabel('Months since cutoff')
    if i == 0:
        ax.set_ylabel('Survival S(t)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Overall Survival: Predicted vs Realized', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'backtest_survival.png', dpi=150, bbox_inches='tight')
plt.show()

---

## CIF Error Heatmaps

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col_idx, model in enumerate(['aj', 'cox', 'rsf']):
    for row_idx, (event, suffix) in enumerate([('Prepayment', '_p'), ('Default', '_d')]):
        ax = axes[row_idx, col_idx]
        pivot = cif_df.pivot(
            index='cutoff', columns='horizon', values=f'{model}_err{suffix}'
        )
        vmax = max(abs(cif_df[f'{model}_err{suffix}']).max(), 0.01)
        sns.heatmap(
            pivot, annot=True, fmt='.4f', cmap='RdBu_r',
            center=0, vmin=-vmax, vmax=vmax, ax=ax,
        )
        ax.set_title(f'{model.upper()} — {event}')
        ax.set_xlabel('Horizon (months)')
        ax.set_ylabel('Cutoff' if col_idx == 0 else '')

plt.suptitle('CIF Prediction Error (Predicted - Realized)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'backtest_cif_error_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Summary Statistics

In [ ]:
# Mean absolute error across all cutoff-horizon combinations
summary_rows = []
for model in ['aj', 'cox', 'rsf']:
    mae_p = cif_df[f'{model}_err_p'].abs().mean()
    mae_d = cif_df[f'{model}_err_d'].abs().mean()
    rmse_p = np.sqrt((cif_df[f'{model}_err_p'] ** 2).mean())
    rmse_d = np.sqrt((cif_df[f'{model}_err_d'] ** 2).mean())
    bias_p = cif_df[f'{model}_err_p'].mean()
    bias_d = cif_df[f'{model}_err_d'].mean()
    summary_rows.append({
        'Model': model.upper(),
        'MAE Prepay': mae_p,
        'MAE Default': mae_d,
        'RMSE Prepay': rmse_p,
        'RMSE Default': rmse_d,
        'Bias Prepay': bias_p,
        'Bias Default': bias_d,
    })

summary_df = pd.DataFrame(summary_rows)
print('=== Model Comparison: CIF Prediction Accuracy ===')
print(summary_df.to_string(index=False, float_format='{:.5f}'.format))

# Per-cutoff summary including training data size
print('\n=== Per-Cutoff Results ===')
for cutoff_str, res in all_results.items():
    real = res['realized']
    print(f'\n{cutoff_str}:')
    print(f'  Training: Cox={res["n_cox_train"]:,} loans, RSF={res["n_rsf_train"]:,} loans')
    print(f'  Test: {res["n_active"]:,} active loans')
    print(f'  Realized: {real["n_prepay"]:,} prepays ({real["n_prepay"]/res["n_active"]*100:.1f}%), '
          f'{real["n_default"]:,} defaults ({real["n_default"]/res["n_active"]*100:.1f}%)')

---

## Conclusions

### Methodology
- **No data leakage**: All models retrained at each cutoff using only pre-cutoff data
- Only information **available at the cutoff date** is used for prediction
- Macro features are **frozen** at cutoff values (no scenario assumptions)
- Only deterministic features (`bal_repaid` from amortization, `t_act_12m`) evolve
- The Aalen-Johansen CIF serves as a **population-level baseline** without loan-level features

### Limitations
- Frozen macro is conservative — in practice you would use scenario projections
- The 2020-06 cutoff captures the COVID refinancing wave, a challenging regime shift
- Default CIF is very small in post-2010 data, making it harder to evaluate default predictions
- RSF training uses terminal observations which may include feature leakage from the event itself